In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [4]:
# ================================================================
# Imports
# ================================================================
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, bmat, diags
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr
from scipy.sparse.csgraph import connected_components

# ================================================================
# Load data (remove isolated points FIRST)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0]

snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_all = snow.iloc[:, :2].to_numpy()
y_all      = snow.iloc[:, 2:].to_numpy()

# ================================================================
# Build adjacency FIRST (on non-isolated set)
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_all[:, 0], coords_all[:, 1]),
    crs="EPSG:4326"
)

gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta = np.arctan2(dif[1], dif[0])
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

rotated = (xy @ R.T) / 1e6
Distances = squareform(pdist(rotated))

Omg = (Distances <= 0.22).astype(int)
np.fill_diagonal(Omg, 0)
Omg = csr_matrix(Omg)

# ================================================================
# Keep TWO largest components
# ================================================================
n_comp, labels = connected_components(Omg, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

# -------------------------
# SUBSET EVERYTHING FIRST
# -------------------------
coords = coords_all[use_idx]
y      = y_all[use_idx]
Omg    = Omg[use_idx][:, use_idx]

S, TT = y.shape
period = 52

print("Using S =", S)

# ================================================================
# Global time trend (scale AFTER subset)
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# Load temperature — subset first, THEN scale
# ================================================================
snow_temp = pyreadr.read_r("snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)

temp_all = snow_temp.drop(index=no_nbs).iloc[:, 2:].to_numpy()
temp = temp_all[use_idx]

temp_mean = temp.mean()
temp_sd   = temp.std(ddof=0)
temp_scaled = (temp - temp_mean) / temp_sd

# ================================================================
# Latitude & elevation — subset first, THEN scale
# ================================================================
lat_raw = coords[:, 1]
lat = (lat_raw - lat_raw.mean()) / lat_raw.std(ddof=1)

elev_raw_all = pd.read_csv("curr_elev.csv").iloc[:, 3].to_numpy()
elev_raw = elev_raw_all[use_idx]
elev = (elev_raw - elev_raw.mean()) / elev_raw.std(ddof=1)

# ================================================================
# Build ICAR precision
# ================================================================
deg = np.array(Omg.sum(axis=1)).flatten()
D = diags(deg)
prec = D - Omg

# ================================================================
# MCMC settings
# ================================================================
burn = 1000
thin = 5
tot_save = 1000

a_tau = 2.0
b_tau = 25.0

# ================================================================
# BYM++factor runner
# ================================================================
def run_bym_factor(event_name, loc_mask, kappa_transform, save_path):

    loc = np.where(loc_mask)
    pairs = np.column_stack(loc)
    pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]
    pairs[:, 1] += 1

    row_idx  = pairs[:, 0]
    time_idx = pairs[:, 1] - 1
    N = len(row_idx)

    next_y = y[pairs[:, 0], pairs[:, 1]]
    kappa  = kappa_transform(next_y)

    # ------------------------------------------------------------
    # Time variables
    # ------------------------------------------------------------
    t_raw   = time_idx + 1                  # for sin/cos
    t_trend = t_trend_full[time_idx]        # subset ONLY

    # ------------------------------------------------------------
    # Covariates (BYM part)
    # ------------------------------------------------------------
    covariates = np.column_stack([
        np.ones(N), np.ones(N),
        np.cos(2*np.pi*t_raw / period),
        np.cos(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        t_trend, t_trend
    ])

    K = covariates.shape[1]   # 8
    theta_dim = K * S + 3     # +3 factor parameters

    # ------------------------------------------------------------
    # BYM design matrix
    # ------------------------------------------------------------
    rows, cols, vals = [], [], []
    for i in tqdm(range(N), desc=f"Design {event_name}"):
        s = row_idx[i]
        for k in range(K):
            rows.append(i)
            cols.append(s + k*S)
            vals.append(covariates[i, k])

    X_bym = coo_matrix((vals, (rows, cols)), shape=(N, K*S)).tocsr()

    # ------------------------------------------------------------
    # Factor design (trend × spatial covariates)
    # ------------------------------------------------------------
    t_lat  = t_trend * lat[row_idx]
    t_elev = t_trend * elev[row_idx]
    t_temp = t_trend * temp_scaled[row_idx, time_idx]

    X_fac = csr_matrix(np.column_stack([t_lat, t_elev, t_temp]))

    from scipy.sparse import hstack

    X = hstack([X_bym, X_fac], format="csr")

    # ------------------------------------------------------------
    # Storage
    # ------------------------------------------------------------
    total_iters = burn + tot_save * thin
    all_theta = np.zeros((theta_dim, tot_save))
    all_tau   = np.zeros((K, tot_save))

    curr_theta = np.zeros(theta_dim)
    curr_tau   = np.ones(K)

    save_idx = 0

    # ------------------------------------------------------------
    # MCMC
    # ------------------------------------------------------------
    for it in tqdm(range(total_iters), desc=f"MCMC {event_name}"):

        phi = X @ curr_theta
        omega = random_polyagamma(1, phi, size=N)

        # ---- prior precision
        block_list = []
        for j in range(K):
            if j % 2 == 0:
                block_list.append((1 / curr_tau[j]) * prec)
            else:
                block_list.append((1 / curr_tau[j]) * diags(np.ones(S)))

        block_list.append((1 / 100) * diags(np.ones(3)))  # factor prior

        blocks = [[block_list[i] if i == j else None
                   for j in range(K + 1)]
                  for i in range(K + 1)]

        curr_prec = bmat(blocks, format="csr")

        XtOmega = X.T.multiply(omega)
        post_prec = XtOmega @ X + curr_prec
        post_prec = post_prec

        factor = cholesky(post_prec, mode="simplicial")

        rhs = X.T @ kappa
        mu = factor.solve_A(rhs)

        # correct Gaussian sampling
        z = np.random.randn(theta_dim)
        z = z / np.sqrt(factor.D())
        z = factor.solve_Lt(z)
        z = factor.apply_Pt(z)

        curr_theta = mu + z

        # ---- update tau
        for j in range(K):
            sl = slice(j*S, (j+1)*S)
            beta = curr_theta[sl]
            quad = beta @ (prec @ beta) if j % 2 == 0 else beta @ beta
            curr_tau[j] = 1 / np.random.gamma(
                a_tau + S/2,
                1 / (b_tau + quad/2)
            )

        if it >= burn and (it - burn) % thin == 0:
            all_theta[:, save_idx] = curr_theta
            all_tau[:, save_idx]   = curr_tau
            save_idx += 1
            if save_idx == tot_save:
                break

    np.savez_compressed(save_path, all_theta=all_theta, all_tau=all_tau)

# ================================================================
# Run p01
# ================================================================
run_bym_factor(
    "p01",
    loc_mask=(y[:, :-1] == 0),
    kappa_transform=lambda ny: ny - 0.5,
    save_path=r"D:\77\Research\temp\snow\bym_factor_01_2pc.npz"
)

# ================================================================
# Run p10
# ================================================================
run_bym_factor(
    "p10",
    loc_mask=(y[:, :-1] == 1),
    kappa_transform=lambda ny: (1 - ny) - 0.5,
    save_path=r"D:\77\Research\temp\snow\bym_factor_10_2pc.npz"
)


Using S = 1557


MCMC p01:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_35144\1438353960.py:230: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(post_prec, mode="simplicial")
MCMC p10: 100%|█████████▉| 5995/6000 [3:13:27<00:09,  1.94s/it]  


In [7]:
# ================================================================
# Posterior Mean LLH
# BYM + Factor (Gamma tau version)
# SCALE FIRST → THEN FILTER COMPONENT
# ================================================================

import numpy as np
import pyreadr
import geopandas as gpd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix
from scipy.sparse.csgraph import connected_components

DIST_TH = 0.22
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1


# ================================================================
# 1️⃣ Load + drop no_nbs
# ================================================================

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_full = snow.iloc[:, :2].to_numpy()
y_full = snow.iloc[:, 2:].to_numpy()

S0, TT = y_full.shape

# ================================================================
# 2️⃣ GLOBAL SCALE FIRST (before filtering component)
# ================================================================

# time
t_full = np.arange(1, TT+1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# temperature
snow_temp = pyreadr.read_r("snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.drop(index=no_nbs).iloc[:,2:].to_numpy()

temp_scaled_full = (temp_full - temp_full.mean()) / temp_full.std(ddof=0)

# lat
lat_raw = coords_full[:,1]
lat_full = (lat_raw - lat_raw.mean()) / lat_raw.std(ddof=1)

# elev
import pandas as pd
elev_raw = pd.read_csv("curr_elev.csv").iloc[:,3].to_numpy()
elev_full = (elev_raw - elev_raw.mean()) / elev_raw.std(ddof=1)


# ================================================================
# 3️⃣ NOW filter two largest components
# ================================================================

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_full[:,0], coords_full[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
W = (squareform(pdist(xy)) <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

keep = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords_full[keep]
y = y_full[keep]
lat = lat_full[keep]
elev = elev_full[keep]
temp_scaled = temp_scaled_full[keep]

S, TT = y.shape


# ================================================================
# 4️⃣ Build design (same as fitting)
# ================================================================

def build_design(transition):

    if transition == "p01":
        loc_mask = (y[:, :-1] == 0)
        success = lambda ny: ny
    else:
        loc_mask = (y[:, :-1] == 1)
        success = lambda ny: 1 - ny

    row_idx, time_idx = np.where(loc_mask)
    pairs = np.column_stack([row_idx, time_idx])
    pairs = pairs[np.lexsort((pairs[:,0], pairs[:,1]))]
    pairs[:,1] += 1

    row_idx = pairs[:,0]
    time_idx = pairs[:,1] - 1

    next_y = y[pairs[:,0], pairs[:,1]]
    y_vec = success(next_y).astype(float)

    t_raw = time_idx + 1
    t_trend = t_trend_full[time_idx]

    cov = np.column_stack([
        np.ones(len(row_idx)), np.ones(len(row_idx)),
        np.cos(2*np.pi*t_raw/period), np.cos(2*np.pi*t_raw/period),
        np.sin(2*np.pi*t_raw/period), np.sin(2*np.pi*t_raw/period),
        t_trend, t_trend
    ])

    K = 8
    eta_dim = K*S + 3

    rows, cols, vals = [], [], []
    for i in range(len(row_idx)):
        s = row_idx[i]
        for k in range(K):
            rows.append(i)
            cols.append(k*S + s)
            vals.append(cov[i,k])

        rows += [i,i,i]
        cols += [K*S, K*S+1, K*S+2]
        vals += [
            t_trend[i] * lat[s],
            t_trend[i] * elev[s],
            t_trend[i] * temp_scaled[s,time_idx[i]]
        ]

    X = coo_matrix((vals,(rows,cols)),
                   shape=(len(row_idx),eta_dim)).tocsr()

    return X, y_vec


# ================================================================
# 5️⃣ Compute LLH
# ================================================================

def compute_llh(file_path, transition):

    data = np.load(file_path)
    all_theta = data["all_theta"]

    M = all_theta.shape[1]

    X, y_vec = build_design(transition)

    llh_draws = np.zeros(M)

    for m in tqdm(range(M), desc=f"LLH {transition}"):

        theta = all_theta[:,m]
        psi = X @ theta

        softplus = np.log1p(np.exp(-np.abs(psi))) + np.maximum(psi,0)
        llh_draws[m] = np.sum(y_vec * psi - softplus)

    return llh_draws.mean()

BASE_DIR = Path(r"D:\77\Research\temp\snow")
llh01 = compute_llh(BASE_DIR/"bym_factor_01_2pc.npz","p01")
llh10 = compute_llh(BASE_DIR/"bym_factor_10_2pc.npz","p10")

print("\nPosterior mean LLH p01:", llh01)
print("Posterior mean LLH p10:", llh10)
print("TOTAL:", llh01 + llh10)

LLH p10: 100%|██████████| 1000/1000 [01:00<00:00, 16.51it/s]


Posterior mean LLH p01: -340826.7925999874
Posterior mean LLH p10: -267978.9258673335
TOTAL: -608805.7184673209
